In [1]:
import pandas as pd
from pathlib import Path

# Load Week 2 Residential datasets
sold = pd.read_csv("../output/week2_sold_residential.csv", low_memory=False)
listings = pd.read_csv("../output/week2_listings_residential.csv", low_memory=False)

# Download weekly mortgage rates from FRED
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url)

# Rename the columns
mortgage.columns = ["date", "rate_30yr_fixed"]

# Convert dates and rates to appropriate types
mortgage["date"] = pd.to_datetime(mortgage["date"], errors="coerce")
mortgage["rate_30yr_fixed"] = pd.to_numeric(
    mortgage["rate_30yr_fixed"], errors="coerce"
)

print("Mortgage records:", len(mortgage))
display(mortgage.head())

Mortgage records: 2899


,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [2]:
# Extract year and month
mortgage["year_month"] = mortgage["date"].dt.to_period("M")

# Calculate average mortgage rate for each month
mortgage_monthly = mortgage.groupby(
    "year_month"
)["rate_30yr_fixed"].mean().reset_index()

print("Monthly mortgage rates:")
display(mortgage_monthly.tail(12))

Monthly mortgage rates:


,year_month,rate_30yr_fixed
655,2025-11,6.2375
656,2025-12,6.1900
657,2026-01,6.1025
658,2026-02,6.0475
659,2026-03,6.1775
660,2026-04,6.3320
661,2026-05,6.4425
662,2026-06,6.4900
663,2026-07,6.5420
664,2026-08,6.6675


In [3]:
# Create year-month keys
sold["year_month"] = pd.to_datetime(
    sold["CloseDate"], errors="coerce"
).dt.to_period("M")

listings["year_month"] = pd.to_datetime(
    listings["ListingContractDate"], errors="coerce"
).dt.to_period("M")

# Merge mortgage rates with MLS datasets
sold_with_rates = sold.merge(
    mortgage_monthly, on="year_month", how="left"
)

listings_with_rates = listings.merge(
    mortgage_monthly, on="year_month", how="left"
)

print("Sold records:", len(sold_with_rates))
print("Listing records:", len(listings_with_rates))

Sold records: 414054
Listing records: 547162


In [4]:
# Check for missing mortgage rates
sold_missing_rates = sold_with_rates["rate_30yr_fixed"].isnull().sum()
listing_missing_rates = listings_with_rates["rate_30yr_fixed"].isnull().sum()

print("Sold records without mortgage rates:", sold_missing_rates)
print("Listing records without mortgage rates:", listing_missing_rates)

# Make sure the merge didn't change the number of records
print("Sold row count matches:", len(sold) == len(sold_with_rates))
print("Listing row count matches:", len(listings) == len(listings_with_rates))

# Preview the results
display(sold_with_rates[
    ["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]
].head())

display(listings_with_rates[
    ["ListingContractDate", "year_month", "ListPrice", "rate_30yr_fixed"]
].head())

# Identify months that did not match a mortgage rate
print("Unmatched sold months:")
print(sold_with_rates.loc[
    sold_with_rates["rate_30yr_fixed"].isna(), "year_month"
].value_counts().sort_index())

print("Unmatched listing months:")
print(listings_with_rates.loc[
    listings_with_rates["rate_30yr_fixed"].isna(), "year_month"
].value_counts().sort_index())

Sold records without mortgage rates: 0
Listing records without mortgage rates: 0
Sold row count matches: True
Listing row count matches: True


,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-26,2024-01,240000.0,6.6425
1,2024-01-05,2024-01,815000.0,6.6425
2,2024-01-05,2024-01,810000.0,6.6425
3,2024-01-02,2024-01,2100000.0,6.6425
4,2024-01-22,2024-01,1950000.0,6.6425


,ListingContractDate,year_month,ListPrice,rate_30yr_fixed
0,2024-01-01,2024-01,1340000.0,6.6425
1,2024-01-24,2024-01,2500000.0,6.6425
2,2024-01-12,2024-01,3150000.0,6.6425
3,2024-01-20,2024-01,3090000.0,6.6425
4,2024-01-12,2024-01,12725000.0,6.6425


Unmatched sold months:
Series([], Freq: M, Name: count, dtype: int64)
Unmatched listing months:
Series([], Freq: M, Name: count, dtype: int64)


In [5]:
# Save datasets with mortgage rates included
Path("../output").mkdir(exist_ok=True)

sold_with_rates.to_csv(
    "../output/sold_with_mortgage_rates.csv",
    index=False
)

listings_with_rates.to_csv(
    "../output/listings_with_mortgage_rates.csv",
    index=False
)

print("Mortgage-enriched datasets saved!")

Mortgage-enriched datasets saved!


## Weeks 2–3: Mortgage Rate Enrichment

### Goal
Add national 30-year fixed mortgage rates to the Residential sold and listing datasets to support future housing market analysis.

### Findings
- Retrieved 2,899 weekly mortgage rate records from FRED (MORTGAGE30US).
- Converted weekly rates into monthly averages.
- Matched sold properties using CloseDate and listings using ListingContractDate.
- Successfully merged mortgage rates into:
  - 414,054 sold records.
  - 547,162 listing records.
- Found 0 missing mortgage rates in either dataset.
- Row counts remained unchanged after both merges.
- Example: January 2024 had an average 30-year mortgage rate of 6.6425%.
- Saved both mortgage-enriched datasets as new CSV files.

### Business Takeaway
The datasets can now be used to explore how mortgage rates relate to housing prices, listing activity, and market demand. Adding monthly mortgage rates allows housing market trends to be compared with borrowing costs over time.

### Note
- Mortgage rates represent national monthly averages, not individual buyers' actual loan rates.
- Sold properties use their closing month, while listings use their listing month.
- The merge was validated successfully, but no relationship between mortgage rates and housing market activity has been established yet.